<div dir="rtl">

# جلسه ۳: مبانی PyTorch: تنسور، Autograd و GPU

**PyTorch Fundamentals: Tensors, Autograd and GPU**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/03.%20PyTorch_Fundamentals.ipynb)

**دوره یادگیری عمیق و شبکه‌های عصبی** — کیوان محمودی

### اهداف این جلسه

- ساخت و دست‌کاری تنسورها (Tensors): نوع داده، ابعاد، Indexing و Broadcasting
- ارتباط PyTorch با NumPy و انتقال محاسبات به GPU
- مشتق‌گیری خودکار با `autograd`: گراف پویا، `backward`، `no_grad` و `detach`
- تطبیق Backpropagation دستی جلسه ۲ با autograd
- ساخت اولین مدل با `nn.Linear`، `nn.MSELoss` و `torch.optim`

> **نکته اجرا:** این نوتبوک روی GPU سریع‌تر اجرا می‌شود. در Colab از منوی `Runtime → Change runtime type` گزینه **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl">

## ۱. چرا PyTorch؟

در جلسه قبل Backpropagation را دستی پیاده کردیم. برای شبکه‌های واقعی با ده‌ها لایه نوشتن دستی گرادیان‌ها عملی نیست. **چارچوب‌های یادگیری عمیق (Deep Learning Frameworks)** سه کار اصلی برای ما انجام می‌دهند:

۱. **محاسبات تنسوری سریع** روی CPU و GPU.

۲. **مشتق‌گیری خودکار (Automatic Differentiation)**؛ همان کاری که کلاس `Value` ما انجام می‌داد، اما برای تنسورها و بسیار بهینه.

۳. **بلوک‌های آماده** برای لایه‌ها، توابع هزینه، بهینه‌سازها و بارگذاری داده.

**PyTorch** که در سال ۲۰۱۶ توسط Meta AI منتشر شد، امروز رایج‌ترین چارچوب در پژوهش است و در صنعت هم بسیار استفاده می‌شود. اغلب مدل‌های Hugging Face هم با PyTorch نوشته شده‌اند. ویژگی مهم آن **گراف محاسباتی پویا (Dynamic Computational Graph)** است: گراف همزمان با اجرای کد پایتون ساخته می‌شود، پس اشکال‌زدایی آن مثل کد پایتون معمولی است.

PyTorch روی Colab از قبل نصب است. برای نصب روی سیستم شخصی، دستور مناسب را از [pytorch.org](https://pytorch.org/get-started/locally/) بردارید.

</div>

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

torch.manual_seed(0)

<div dir="rtl">

## ۲. تنسور (Tensor)

تنسور ساختار داده اصلی PyTorch است: یک آرایه چندبعدی شبیه `ndarray` در NumPy، با دو قابلیت اضافه: اجرا روی **GPU** و ثبت تاریخچه محاسبات برای **مشتق‌گیری خودکار**.

| مرتبه (Rank) | نام | مثال |
|---|---|---|
| 0 | اسکالر (Scalar) | مقدار Loss |
| 1 | بردار (Vector) | بایاس یک لایه |
| 2 | ماتریس (Matrix) | وزن‌های یک لایه Linear یا دسته‌ای از بردارها |
| 3 | تنسور مرتبه ۳ | دسته‌ای از جمله‌ها: `(batch, seq_len, embedding_dim)` |
| 4 | تنسور مرتبه ۴ | دسته‌ای از تصاویر: `(batch, channels, height, width)` |

</div>

In [ ]:
# creating tensors
a = torch.tensor([[1, 2, 3], [4, 5, 6]])          # from a Python list
b = torch.zeros(2, 3)
c = torch.ones(2, 3, dtype=torch.float64)
d = torch.arange(0, 10, 2)
e = torch.linspace(0, 1, 5)
r = torch.randn(3, 4)                                # standard normal
i = torch.randint(0, 10, (2, 2))

for name, t in [('a', a), ('b', b), ('c', c), ('d', d), ('e', e), ('r', r), ('i', i)]:
    print(f'{name}: shape={tuple(t.shape)}, dtype={t.dtype}')

<div dir="rtl">

### نوع داده (dtype)

نوع پیش‌فرض اعداد اعشاری در PyTorch `float32` است (در NumPy `float64`). شبکه‌های عصبی تقریباً همیشه با `float32` (یا حتی `float16`/`bfloat16` برای سرعت بیشتر) کار می‌کنند. برچسب‌های کلاس برای `CrossEntropyLoss` باید از نوع `int64` (`torch.long`) باشند. بسیاری از خطاهای رایج در PyTorch ناشی از **ناسازگاری نوع داده** یا **ناسازگاری ابعاد** هستند.

</div>

In [ ]:
x = torch.tensor([1.5, 2.5])
print(x.dtype)                       # float32
print(x.to(torch.int64))             # cast
print(x.long(), x.double(), x.half())

<div dir="rtl">

### تغییر شکل (Reshaping) و Indexing

- `view` و `reshape` شکل تنسور را عوض می‌کنند (`-1` یعنی «این بعد را خودت حساب کن»).
- `unsqueeze` یک بعد با اندازه ۱ اضافه و `squeeze` آن را حذف می‌کند.
- `permute` ترتیب ابعاد را عوض می‌کند (مثلاً تبدیل تصویر از `HWC` به `CHW`).
- Indexing و Slicing دقیقاً مثل NumPy است.

</div>

In [ ]:
t = torch.arange(24)
print('original      :', t.shape)
print('view(2, 3, 4) :', t.view(2, 3, 4).shape)
print('reshape(4, -1):', t.reshape(4, -1).shape)

img = torch.rand(32, 32, 3)                     # height, width, channels (e.g. from PIL / NumPy)
img_chw = img.permute(2, 0, 1)                  # channels, height, width (what PyTorch expects)
batch = img_chw.unsqueeze(0)                    # add a batch dimension
print('HWC -> CHW -> NCHW:', img.shape, '->', img_chw.shape, '->', batch.shape)
print('squeeze back      :', batch.squeeze(0).shape)

m = torch.arange(12).view(3, 4)
print(m)
print('row 1        :', m[1])
print('column 2     :', m[:, 2])
print('values > 5   :', m[m > 5])

<div dir="rtl">

### عملیات ریاضی و Broadcasting

عملیات عنصربه‌عنصر (`+`، `*`، `torch.exp`، …)، کاهشی (`sum`، `mean`، `max` در امتداد یک بعد با آرگومان `dim`) و ضرب ماتریسی (`@` یا `torch.matmul`) مثل NumPy کار می‌کنند. **Broadcasting** اجازه می‌دهد تنسورهایی با ابعاد متفاوت (اما سازگار) با هم ترکیب شوند؛ مثلاً اضافه کردن بردار بایاس به همه سطرهای یک دسته:

</div>

In [ ]:
X = torch.randn(5, 3)          # batch of 5 samples, 3 features
W = torch.randn(3, 2)
bias = torch.randn(2)

out = X @ W + bias             # (5,3) @ (3,2) -> (5,2), bias broadcast over the 5 rows
print('output shape:', out.shape)
print('mean over the batch (dim=0):', out.mean(dim=0))
print('argmax per sample  (dim=1):', out.argmax(dim=1))

<div dir="rtl">

### ارتباط با NumPy

`torch.from_numpy` و `.numpy()` تبدیل را **بدون کپی** انجام می‌دهند؛ یعنی هر دو شیء **حافظه مشترک** دارند و تغییر یکی دیگری را هم تغییر می‌دهد. اگر کپی مستقل لازم است از `torch.tensor(arr)` یا `.clone()` استفاده کنید.

</div>

In [ ]:
arr = np.array([1.0, 2.0, 3.0])
shared = torch.from_numpy(arr)
copied = torch.tensor(arr)
arr[0] = 100.0
print('from_numpy (shared memory):', shared)
print('torch.tensor (copy)       :', copied)

<div dir="rtl">

## ۳. محاسبات روی GPU

GPU هزاران هسته ساده دارد که ضرب ماتریسی را به‌صورت موازی انجام می‌دهند. برای استفاده از آن کافی است تنسورها (و بعداً مدل) را با `.to(device)` به GPU منتقل کنیم. **قانون مهم:** همه تنسورهایی که در یک عملیات شرکت می‌کنند باید روی یک دستگاه باشند؛ در غیر این صورت خطای `Expected all tensors to be on the same device` می‌گیریم.

مقایسه سرعت یک ضرب ماتریسی بزرگ (اگر GPU فعال نباشد، فقط زمان CPU نمایش داده می‌شود):

</div>

In [ ]:
def time_matmul(device, n=4096, repeats=5):
    A = torch.randn(n, n, device=device)
    B = torch.randn(n, n, device=device)
    A @ B                                       # warm-up
    if device.type == 'cuda':
        torch.cuda.synchronize()                # GPU calls are asynchronous: wait before timing
    start = time.perf_counter()
    for _ in range(repeats):
        A @ B
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / repeats


cpu_time = time_matmul(torch.device('cpu'))
print(f'CPU: {cpu_time * 1000:.1f} ms per 4096x4096 matmul')
if device.type == 'cuda':
    gpu_time = time_matmul(device)
    print(f'GPU: {gpu_time * 1000:.1f} ms  ->  {cpu_time / gpu_time:.0f}x faster')

<div dir="rtl">

## ۴. مشتق‌گیری خودکار (Autograd)

اگر برای یک تنسور `requires_grad=True` تنظیم شود، PyTorch همه عملیات روی آن را در یک گراف محاسباتی ثبت می‌کند. با صدا زدن `.backward()` روی یک خروجی اسکالر، گرادیان آن نسبت به همه تنسورهای برگ (Leaf) در ویژگی `.grad` ذخیره می‌شود؛ دقیقاً همان کاری که کلاس `Value` در جلسه ۲ انجام می‌داد.

همان مثال جلسه قبل، $L = (a\cdot b + c)^2$:

</div>

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(-3.0, requires_grad=True)
c = torch.tensor(10.0, requires_grad=True)

L = (a * b + c) ** 2
L.backward()

print('L =', L.item())
print('dL/da =', a.grad.item(), ' dL/db =', b.grad.item(), ' dL/dc =', c.grad.item())
print('grad_fn of L:', L.grad_fn)          # the last operation in the graph

<div dir="rtl">

همان اعداد جلسه قبل ($-24$، $16$ و $8$) به دست آمد. سه نکته مهم درباره autograd:

**۱. انباشت گرادیان:** گرادیان‌ها مثل کلاس `Value` جمع می‌شوند، نه جایگزین. به همین دلیل در حلقه آموزش قبل از هر `backward` گرادیان‌ها را صفر می‌کنیم.

</div>

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
for i in range(3):
    loss = w ** 2               # d(loss)/dw = 2w = 6
    loss.backward()
    print(f'after backward #{i + 1}: w.grad = {w.grad.item()}')
w.grad.zero_()
print('after zero_()           : w.grad =', w.grad.item())

<div dir="rtl">

**۲. `torch.no_grad()`:** هنگام ارزیابی مدل یا به‌روزرسانی دستی وزن‌ها نیازی به ساخت گراف نداریم. این بلوک مصرف حافظه را کم و سرعت را زیاد می‌کند.

**۳. `detach()`:** یک تنسور جدید با همان داده اما **جدا از گراف** برمی‌گرداند. برای ثبت مقدار Loss یا رسم نمودار از `.item()` (برای اسکالر) یا `.detach().cpu().numpy()` استفاده می‌کنیم.

</div>

In [ ]:
x = torch.randn(3, requires_grad=True)
y = x * 2
print('y.requires_grad:', y.requires_grad)

with torch.no_grad():
    z = x * 2
print('inside no_grad, z.requires_grad:', z.requires_grad)

print('detached:', y.detach().requires_grad, '| as NumPy:', y.detach().cpu().numpy())

<div dir="rtl">

### بررسی Backpropagation دستی جلسه ۲ با autograd

گرادیان‌هایی که برای شبکه دولایه دستی استخراج کردیم را با autograd مقایسه می‌کنیم. اگر هر دو یکی باشند، هم فرمول‌های ما و هم درک ما از autograd درست است:

</div>

In [ ]:
torch.manual_seed(0)
N, d_in, d_hidden, n_classes = 8, 4, 6, 3
X = torch.randn(N, d_in)
y = torch.randint(0, n_classes, (N,))

W1 = torch.randn(d_in, d_hidden, requires_grad=True)
b1 = torch.zeros(d_hidden, requires_grad=True)
W2 = torch.randn(d_hidden, n_classes, requires_grad=True)
b2 = torch.zeros(n_classes, requires_grad=True)

# forward + autograd backward
Z1 = X @ W1 + b1
H = torch.relu(Z1)
Z2 = H @ W2 + b2
loss = torch.nn.functional.cross_entropy(Z2, y)    # softmax + negative log-likelihood
loss.backward()

# manual backward (formulas from session 2)
with torch.no_grad():
    P = torch.softmax(Z2, dim=1)
    Y = torch.nn.functional.one_hot(y, n_classes).float()
    dZ2 = (P - Y) / N
    dW2 = H.T @ dZ2
    dZ1 = (dZ2 @ W2.T) * (Z1 > 0)
    dW1 = X.T @ dZ1

print('W2 grads match:', torch.allclose(dW2, W2.grad, atol=1e-6))
print('W1 grads match:', torch.allclose(dW1, W1.grad, atol=1e-6))
print('b1 grads match:', torch.allclose(dZ1.sum(0), b1.grad, atol=1e-6))

<div dir="rtl">

## ۵. اولین مدل: رگرسیون خطی با PyTorch

یک مسئله ساده می‌سازیم: داده‌هایی که از رابطه $y = 3x - 2$ به‌علاوه نویز تولید شده‌اند. مدل را دو بار آموزش می‌دهیم: یک بار فقط با تنسورها و autograd، و یک بار با ابزارهای استاندارد PyTorch.

### نسخه ۱: تنسورها و autograd

</div>

In [ ]:
torch.manual_seed(42)
x = torch.linspace(-2, 2, 200).unsqueeze(1)          # shape (200, 1)
y = 3 * x - 2 + 0.4 * torch.randn_like(x)

w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1

for epoch in range(100):
    y_hat = x * w + b
    loss = ((y_hat - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():                           # the update itself must not be tracked
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()
    b.grad.zero_()

print(f'learned: w = {w.item():.3f}, b = {b.item():.3f}   (true: w = 3, b = -2)')

<div dir="rtl">

### نسخه ۲: `nn.Module`، تابع هزینه و Optimizer آماده

در عمل سه بخش تکراری کد بالا را با ابزارهای استاندارد جایگزین می‌کنیم:

| کار | دستی | PyTorch |
|---|---|---|
| تعریف پارامترها و Forward | `x * w + b` | `nn.Linear(1, 1)` |
| تابع هزینه | `((y_hat - y) ** 2).mean()` | `nn.MSELoss()` |
| به‌روزرسانی و صفر کردن گرادیان | `w -= lr * w.grad` | `optimizer.step()` و `optimizer.zero_grad()` |

این همان الگوی **پنج‌مرحله‌ای حلقه آموزش** است که از این به بعد همیشه استفاده می‌کنیم:

</div>

In [ ]:
import torch.nn as nn

model = nn.Linear(in_features=1, out_features=1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

losses = []
for epoch in range(100):
    y_hat = model(x)                   # 1. forward pass
    loss = criterion(y_hat, y)         # 2. compute the loss
    optimizer.zero_grad()              # 3. reset accumulated gradients
    loss.backward()                    # 4. backpropagation
    optimizer.step()                   # 5. update the parameters
    losses.append(loss.item())

print(f'learned: w = {model.weight.item():.3f}, b = {model.bias.item():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses); axes[0].set_title('MSE loss'); axes[0].set_xlabel('epoch')
axes[1].scatter(x, y, s=8, alpha=0.5, label='data')
with torch.no_grad():
    axes[1].plot(x, model(x), color='C1', lw=3, label='model')
axes[1].legend(); axes[1].set_title('fitted line')
plt.show()

<div dir="rtl">

در جلسه بعد همین الگو را با `nn.Module` سفارشی، `Dataset` و `DataLoader` برای آموزش یک شبکه عصبی واقعی روی تصاویر به کار می‌بریم.

## ۶. جمع‌بندی

- تنسور آرایه چندبعدی PyTorch است؛ به `dtype`، `shape` و `device` آن همیشه دقت کنید.
- `view`/`reshape`، `unsqueeze`/`squeeze` و `permute` پرکاربردترین عملیات تغییر شکل هستند.
- `from_numpy` حافظه را با NumPy به اشتراک می‌گذارد.
- برای استفاده از GPU، مدل و داده را با `.to(device)` روی یک دستگاه قرار دهید.
- با `requires_grad=True` و `backward()` گرادیان‌ها خودکار محاسبه می‌شوند؛ گرادیان‌ها انباشته می‌شوند پس باید صفر شوند.
- `torch.no_grad()` برای ارزیابی و `detach()`/`item()` برای جدا کردن مقدار از گراف.
- حلقه آموزش: forward ← loss ← zero_grad ← backward ← step.

## ۷. تمرین

۱. یک دسته ۱۶تایی تصویر رنگی ۶۴×۶۴ با شکل `(16, 64, 64, 3)` بسازید و آن را به شکل مورد انتظار PyTorch یعنی `(16, 3, 64, 64)` تبدیل کنید. سپس میانگین هر کانال رنگی را روی کل دسته حساب کنید (خروجی باید ۳ عدد باشد).

۲. با autograd مشتق تابع $f(x) = \sin(x)\,e^{-x^2}$ را در ۱۰۰ نقطه حساب کنید و با مشتق تحلیلی مقایسه کنید. (راهنمایی: `f(x).sum().backward()`.)

۳. در نسخه ۱ رگرسیون خطی، خط `w.grad.zero_()` را حذف کنید. چه اتفاقی می‌افتد و چرا؟

۴. مدل رگرسیون را برای داده $y = 0.5x^3 - x + \text{noise}$ آموزش دهید. یک بار با `nn.Linear(1, 1)` و یک بار با ویژگی‌های `[x, x², x³]` و `nn.Linear(3, 1)`. نتیجه را مقایسه کنید.

</div>